# A2 · Decisión ZAP (cielo)

**Spec:** [`docs/spec_A2_codex_sky_zap.md`](../docs/spec_A2_codex_sky_zap.md)  |  **Bloque:** A · Reducción  |  **Run de este set:** `ROXs12b_realigned`

Decide y aplica (o descarta) la sustracción de cielo con ZAP.

| | |
|---|---|
| **Entrada** | Cubo reducido |
| **Salida (QC/productos)** | Cubo con cielo tratado (sin QC separado en este run) |
| **Consume aguas abajo** | A3, A4 |


## Qué es ZAP y por qué se necesita

**ZAP** (*Zurich Atmosphere Purge*, Soto et al. 2016) es una sustracción de **residuos de cielo** para MUSE basada en PCA. El pipeline (`muse_scipost subtract_sky`) ya resta un modelo de cielo, pero el **airglow** (líneas de OH y [O I] atmosféricas) es intenso y **varía en el tiempo** entre la exposición de ciencia y el modelo → suelen quedar **residuos de skylines**. ZAP construye una base PCA con los spaxels de **cielo** (con las fuentes enmascaradas) y elimina las componentes que describen esos residuos, dejando la señal astrofísica.

**Por qué importa aquí:** un residuo de skyline mal restado puede **imitar o contaminar** una línea espectral. Buscamos una línea débil de Hα en el compañero, así que el cielo residual es un contaminante de primer orden.

**El peligro (por qué NO se aplica a ciegas):** ZAP necesita suficientes spaxels de cielo *reales*. En el **campo diminuto de NFM**, con una estrella brillante y su compañero, la fracción de cielo es baja y las eigencomponentes pueden **absorber señal del compañero** — incluso *fabricar o borrar* una línea en Hα. Regla del proyecto: ante la duda, **no tocar la señal**.

**Decisión pre-registrada** (`musepipe.reduction.sky_zap.classify_zap_decision`), por métrica, no por juicio. `R` = RMS mediano en ventanas de skyline ÷ RMS mediano en continuo, medido en aperturas de cielo vacías:

| Condición | Decisión |
|---|---|
| `R ≤ 1.5` | **no necesario** → `zap_applied = False` |
| `R > 2.0` | **necesario** → `zap_applied = True` |
| `1.5 < R ≤ 2.0` | zona gris → checkpoint (no aplicar, preguntar) |
| fracción de cielo `< 0.25` | cielo insuficiente → checkpoint (no aplicar) |

Los parámetros de ZAP quedan en *default*; no se itera buscando 'el mejor resultado'.


## Cómo ejecutar de forma independiente

Etapa de **reducción**: la celda de abajo resuelve el comando real para **este objeto** a partir de su `chain.reduction_profile` y de su config, y puede lanzarlo. Son trabajos largos (ver coste), así que se lanzan en segundo plano con el log a la vista; el notebook no se bloquea.

Si algún dato no está declarado en el config del run, la celda lo dice y **no lanza** en vez de inventarse una ruta.

Comando histórico de referencia:

```bash
conda activate MUSE
bash scripts/sky_zap.sh
```


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
cmd, target_run, missing = nb.launch_command('A2', RUN_ID)
print('run que ejecuta esta etapa:', target_run)
print('comando resuelto para este objeto:')
print('   ', cmd or '(sin plantilla)')
if missing:
    print()
    print('NO se puede lanzar: faltan datos en el config del run.')
    print('   sin resolver:', ', '.join(missing))
    print(f'   declara esas claves en runs/{target_run}/config/config.json')

RUN = False   # -> True para LANZAR (trabajo largo: revisa el coste arriba)

if RUN and not missing:
    import subprocess, time
    from pathlib import Path
    log = Path(nb.run_dir(target_run)) / 'logs' / f'a2_launch.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, shell=True, cwd=str(nb.project_root()),
                                stdout=fh, stderr=subprocess.STDOUT)
    print(f'lanzado en segundo plano (pid {proc.pid}); log -> {log}')
    print('sigue el progreso con:  !tail -f', log)
elif RUN:
    print('RUN=True pero hay datos sin resolver: no se lanza nada.')
else:
    print()
    print('Modo auditoría (RUN=False): abajo se carga el QC existente.')


## Resultados que llevaron a la conclusión

Métrica **M4 de cielo** del QC del cubo (`stages/stage00q_qc.json`) aplicada a la regla de decisión pre-registrada.


In [ ]:
qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
m4 = qc.get('m4_sky', {})
R = m4.get('R')
sky_frac = m4.get('sky_fraction')   # QCs antiguos no lo persisten (gap documentado)
print(f'm4_sky.R = {R}   (RMS skyline / RMS continuo en aperturas vacías)   [status {m4.get("status")}]')

# Decisión con la implementación OFICIAL (no reimplementada aquí):
try:
    from musepipe.reduction.sky_zap import classify_zap_decision
    if R is None:
        print('sin dato de R en el QC -> decisión no evaluable')
    else:
        if sky_frac is None:
            print('AVISO: el QC no persiste sky_fraction -> la rama '
                  'insufficient_sky (<0.25) no es re-derivable aquí; se evalúa solo la rama R.')
        d = classify_zap_decision(float(R), float(sky_frac) if sky_frac is not None else 1.0)
        print(f'classify_zap_decision: {d.decision}  ->  zap_applied = {d.zap_applied}'
              f'  (checkpoint_required={d.checkpoint_required})')
except Exception as e:
    print('No se pudo importar musepipe (kernel sin la pila científica):', type(e).__name__, e)
    print('Regla pre-registrada (espejo de classify_zap_decision): R<=1.5 no necesario | '
          'R>2.0 necesario | zona gris -> checkpoint | sky_fraction<0.25 -> checkpoint')

print()
print('Corroboración (nota del QC):')
print('  ', qc.get('note'))
print('M1/M2 se midieron del SKY_SPECTRUM cacheado (airglow, 32 exp,',
      qc.get('m1_wavelength', {}).get('n_measurements'), 'medidas) porque el')
print('cubo restado de cielo tiene <8 skylines usables.')


## De dónde sale R: los datos y la zona

**Un solo FITS:** `cube_telcorr.fits` (el producto de A1), extensión **DATA** (la STAT no interviene en R). R se mide sobre los **spaxels de cielo vacíos** de ese cubo — no hay varios archivos. *(Los 32 `SKY_SPECTRUM` cacheados son de M1/M2, no de R.)*

El gráfico reproduce M4 con las funciones canónicas (`compute_sky_residual_metrics`, `SKYLINE_WINDOWS`, `CONTINUUM_WINDOWS`):

- **Izquierda:** RMS por canal en las aperturas de cielo vs λ. En rojo las ventanas de skyline, en verde las de continuo; las punteadas son las medianas cuyo cociente es `R`. Se ven los residuos de OH en el rojo (>7200 Å), pero su mediana queda **por debajo** del continuo → R < 1.
- **Centro:** la zona de cielo usada (azul) sobre la luz-blanca, **con los ejes en píxeles** y el radio en arcsec marcado desde la primaria; el halo AO queda excluido.
- **Derecha:** la **mediana del cielo por spaxel** vs λ — el nivel que queda *después* de que el DRS restara el cielo. No es cero: si sale sistemáticamente negativo, el DRS sobre-restó, y ese suelo es el que hay que quitar antes de integrar flujo en aperturas grandes (es exactamente lo que mide `scripts/measure_growth_curve.py` para `apcorr`).

### Cómo se decide dónde está el cielo

El criterio de la etapa es `build_source_mask` (`musepipe/reduction/sky_zap.py`), y define **fuente**, no cielo — el cielo es lo que sobra:

1. **Umbral de brillo.** Sobre la luz-blanca: `fondo = mediana` de todo el campo, `σ = robust_sigma`, y se marca como fuente todo lo que supere `fondo + 3σ` (`threshold_sigma`).
2. **Dilatación** de 2 px (`dilation_px`), para no dejar el borde de cada fuente justo fuera de la máscara.
3. **Discos forzados** en las posiciones declaradas (`SourceRegion`). Con `auto_halo=True` el radio no es fijo: `estimate_halo_radius` mide el perfil radial y crece hasta que el brillo cae a `fondo + 1σ` medido en `r > 0.65·r_max`, más un margen de 5 px. Así el disco sigue al halo AO en vez de suponerlo.
4. **Cielo = spaxels finitos NO enmascarados.** `sky_fraction` (0.7607) es esa fracción, y la etapa exige ≥ 0.25: por debajo no hay cielo suficiente para que ZAP aprenda nada.

Es deliberadamente **conservador**: prefiere llamar fuente a un spaxel dudoso antes que meter halo en el cielo, porque contaminar el cielo sesga todo lo que venga después.

> Necesita el kernel **MUSE** (astropy) y el cubo en disco. Usa una máscara de cielo aproximada (percentil 30 de flujo) en vez de la de `build_source_mask`, así que el R reproducido aquí difiere levemente del oficial (0.547, máscara de A4); lo que importa es si cae del mismo lado del umbral `R ≤ 1.5`.


In [ ]:
try:
    MAKE_PLOT = True   # carga el cubo (~3.3 GB) vía astropy; requiere kernel MUSE
    if MAKE_PLOT:
        try:
            import numpy as np
            import matplotlib.pyplot as plt
            from astropy.io import fits
            from musepipe.reduction.sky_zap import (
                compute_sky_residual_metrics, SKYLINE_WINDOWS, CONTINUUM_WINDOWS)

            qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
            cube_path = qc.get('input_cube') or qc.get('cube', {}).get('file')
            print('FITS usado:', cube_path)

            h = fits.open(cube_path, memmap=True)
            data = np.asarray(h[1].data, dtype=np.float32)
            hd = h[1].header
            n3 = hd['NAXIS3']
            wave = hd['CRVAL3'] + (np.arange(n3) - (hd['CRPIX3'] - 1)) * hd['CD3_3']

            wl = np.nanmedian(data, axis=0)
            finite = np.isfinite(wl)
            thr = np.nanpercentile(wl[finite], 30)
            sky_mask = finite & (wl < thr)   # cielo vacio ~ spaxels mas debiles

            m = compute_sky_residual_metrics(data.astype(np.float64), wave, sky_mask)
            R = m['R_skyline_over_continuum']
            rms = m['channel_rms']
            print(f'R reproducido = {R:.3f}   (oficial m4_sky.R = {qc.get("m4_sky", {}).get("R")})')

            # Mediana del cielo POR SPAXEL, canal a canal: el nivel que queda
            # despues de la resta de cielo del DRS. Se calcula sobre los mismos
            # spaxels que la mascara de cielo, por trozos para no duplicar el cubo.
            sky_med = np.full(n3, np.nan)
            _idx = np.where(sky_mask.ravel())[0]
            for _c0 in range(0, n3, 200):
                _blk = data[_c0:_c0 + 200].reshape(min(200, n3 - _c0), -1)[:, _idx]
                sky_med[_c0:_c0 + _blk.shape[0]] = np.nanmedian(_blk, axis=1)
            _fin = np.isfinite(sky_med)
            print(f'mediana del cielo por spaxel: {np.median(sky_med[_fin]):+.3f}'
                  f'  (azul {np.median(sky_med[_fin & (wave < 6000)]):+.3f},'
                  f' rojo {np.median(sky_med[_fin & (wave > 8000)]):+.3f})'
                  f' en unidades del cubo')

            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 4.2),
                                                gridspec_kw={'width_ratios': [2, 1.15, 1.5]})
            ax1.plot(wave, rms, lw=0.5, color='0.35')
            for i, (a, b) in enumerate(SKYLINE_WINDOWS):
                ax1.axvspan(a, b, color='tab:red', alpha=0.18,
                            label='ventana skyline' if i == 0 else None)
            for i, (a, b) in enumerate(CONTINUUM_WINDOWS):
                ax1.axvspan(a, b, color='tab:green', alpha=0.25,
                            label='ventana continuo' if i == 0 else None)
            ax1.axhline(m['skyline_rms_median'], color='tab:red', ls='--', lw=1)
            ax1.axhline(m['continuum_rms_median'], color='tab:green', ls='--', lw=1)
            ax1.set_xlabel('λ [Å]'); ax1.set_ylabel('RMS por canal (aperturas de cielo)')
            ax1.set_title(f'M4: RMS vs λ  →  R = med(skyline)/med(continuo) = {R:.3f}')
            ax1.set_ylim(0, np.nanpercentile(rms, 99)); ax1.legend(fontsize=8)

            # Ejes en PIXELES (antes iba `axis('off')`): sin escala no se puede
            # juzgar a que distancia de la primaria empieza a considerarse cielo.
            _ny, _nx = wl.shape
            ax2.imshow(np.log10(np.clip(wl, 1, None)), origin='lower', cmap='gray',
                       extent=[-0.5, _nx - 0.5, -0.5, _ny - 0.5])
            ov = np.zeros((*wl.shape, 4)); ov[sky_mask] = [0.1, 0.5, 1.0, 0.5]
            ax2.imshow(ov, origin='lower', extent=[-0.5, _nx - 0.5, -0.5, _ny - 0.5])
            _sy, _sx = np.unravel_index(np.nanargmax(wl), wl.shape)
            _pix = abs(float(hd.get('CD1_1') or hd.get('CDELT1', 0.0))) * 3600.0
            if _pix > 0:
                for _as in (0.5, 1.0, 2.0):
                    _r = _as / _pix
                    ax2.add_patch(plt.Circle((_sx, _sy), _r, fill=False, color='tab:orange',
                                             lw=0.8, ls='--'))
                    ax2.annotate(f'{_as:g}"', (_sx, _sy + _r), color='tab:orange',
                                 fontsize=6.5, ha='center', va='bottom')
            ax2.plot(_sx, _sy, '+', color='tab:cyan', ms=8, mew=1.4)
            ax2.set_xlabel('x [px]'); ax2.set_ylabel('y [px]')
            ax2.tick_params(labelsize=7)
            _esc = f' · {_pix:.4f}"/px' if _pix > 0 else ''
            ax2.set_title(f'Zona de cielo (azul) sobre luz-blanca{_esc}', fontsize=9)

            ax3.plot(wave, sky_med, lw=0.5, color='tab:purple')
            ax3.axhline(0, color='tab:red', ls='--', lw=1.0, label='cero (cielo perfecto)')
            _mm = np.median(sky_med[_fin])
            ax3.axhline(_mm, color='0.35', ls=':', lw=1.0,
                        label=f'mediana global {_mm:+.3f}')
            _q = np.nanpercentile(sky_med[_fin], [1, 99])
            ax3.set_ylim(_q[0] - 0.2 * abs(_q[0]), _q[1] + 0.2 * abs(_q[1]))
            ax3.set_xlabel('λ [Å]'); ax3.set_ylabel('mediana del cielo por spaxel')
            ax3.legend(fontsize=7)
            ax3.set_title('lo que queda tras la resta de cielo del DRS', fontsize=9)
            fig.tight_layout()

            outdir = nb.run_dir(RUN_ID) / 'plots' / 'a2_m4'
            outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / 'm4_R.png', dpi=110)
            print('figura ->', outdir / 'm4_R.png')
            plt.show()
            h.close()
        except Exception as e:
            print('No se pudo generar el plot:', type(e).__name__, e)
            print('Necesita el kernel MUSE (astropy) y el cubo en disco (campo input_cube del QC).')
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## Decisiones y notas
- La métrica M4 (`R`) y la caracterización del airglow viven en A4/`stage00q_qc.json`; la regla de decisión, en `musepipe/reduction/sky_zap.py`.


## Conclusión (registrada)

**Decisión de este objeto: `zap_applied = False` (`not_needed`), con R = 0.6169 sobre la máscara de A2.** `n/d` = A2 no ha corrido para esta cadena: la decisión de abajo aún no está registrada para el objeto.

- **Datos:** cubo NFM-AO auto-reducido `cube_telcorr.fits` + `SKY_SPECTRUM` cacheado para caracterizar el airglow (el detalle del OB y las exposiciones de **este** objeto sale del QC de A1/A4; la celda de setup imprime de qué run vienen).
- **Evidencia:** `R = 0.547` frente al umbral `1.5` (estado M4 = **yellow**); el cubo restado de cielo tiene pocas skylines usables (residual al nivel de ruido). En el campo diminuto NFM, ZAP aportaría ~0 y arriesgaría absorber señal del compañero.
- **Salvedad de M4:** el residuo es bajo, pero la escasez de skylines hace la métrica menos robusta que en WFM. No bloqueante.
- **Para el paper:** registrar como decisión con su métrica (R), **no** como omisión. Nada es paper-válido hasta cerrar el A-block del objeto.
